In [ ]:
import os
import gc
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
)
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras.layers import Input, GRU, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras import mixed_precision

# ----------------------------
# Configuration
# ----------------------------
CSV_PATH = "Sepsis Prediction Dataset.csv"
TARGET = "SepsisLabel"
GROUP = "Patient_ID"
HOUR_COL = "Hour"

N_STEPS = 24
PREDICTION_WINDOW = 6

TEST_SIZE = 0.20
VAL_SIZE = 0.10
RANDOM_STATE = 42

BATCH_SIZE = 32
EPOCHS = 20
PATIENCE = 5
LEARNING_RATE = 1e-3

USE_MISSING_INDICATORS = False
MAX_TRAIN_SAMPLES = 300_000
SHUFFLE_BUFFER = 20_000

# Drop feature columns that are almost entirely missing.
MAX_COL_MISSING = 0.98


def set_seed(seed: int = 42) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

class CausalPreprocessor:
    """
    Causal preprocessing:
    - Fit on train split only.
    - No backward fill (no future leakage).
    - Forward fill within patient, then train-median imputation.
    - Add per-hour diffs and missingness indicators.
    - Scale only continuous features on train split.
    """

    def __init__(
        self,
        target_col: str = TARGET,
        group_col: str = GROUP,
        hour_col: str = HOUR_COL,
        max_col_missing: float = MAX_COL_MISSING,
    ):
        self.target_col = target_col
        self.group_col = group_col
        self.hour_col = hour_col
        self.max_col_missing = max_col_missing

        self.base_cols = None
        self.diff_cols = None
        self.miss_cols = None
        self.scale_cols = None
        self.feature_cols = None
        self.medians = None
        self.scaler = None

    def _sort(self, df: pd.DataFrame) -> pd.DataFrame:
        return df.sort_values([self.group_col, self.hour_col]).reset_index(drop=True)

    def fit(self, df_train: pd.DataFrame):
        df_train = self._sort(df_train.copy())

        num_cols = df_train.select_dtypes(include=[np.number]).columns.tolist()
        exclude = {self.target_col, self.group_col, self.hour_col}
        candidate = [c for c in num_cols if c not in exclude]

        missing_rate = df_train[candidate].isna().mean()
        self.base_cols = [c for c in candidate if missing_rate[c] <= self.max_col_missing]
        if not self.base_cols:
            raise ValueError("No usable numeric feature columns found after missing-rate filtering.")

        # Causal forward fill only
        work = df_train[[self.group_col] + self.base_cols].copy()
        for c in self.base_cols:
            work[c] = work.groupby(self.group_col, sort=False)[c].ffill()

        # Train medians (after forward fill)
        self.medians = work[self.base_cols].median(numeric_only=True)
        work[self.base_cols] = work[self.base_cols].fillna(self.medians)

        self.diff_cols = [f"{c}_diff" for c in self.base_cols]
        diffs = work.groupby(self.group_col, sort=False)[self.base_cols].diff().fillna(0.0)
        diffs.columns = self.diff_cols

        self.miss_cols = [f"{c}_missing" for c in self.base_cols] if USE_MISSING_INDICATORS else []
        self.scale_cols = self.base_cols + self.diff_cols

        scale_frame = pd.concat([work[self.base_cols], diffs], axis=1).astype(np.float32)
        self.scaler = StandardScaler()
        self.scaler.fit(scale_frame[self.scale_cols])

        self.feature_cols = self.scale_cols + self.miss_cols
        return self

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        if self.scaler is None:
            raise RuntimeError("Preprocessor is not fitted. Call fit() on training data first.")

        out = self._sort(df.copy())
        for c in self.base_cols:
            if c not in out.columns:
                out[c] = np.nan

        # Missingness indicators from raw values before fill
        miss_ind = None
        if USE_MISSING_INDICATORS:
            miss_ind = out[self.base_cols].isna().astype(np.float32)
            miss_ind.columns = self.miss_cols

        # Causal fill + median imputation
        for c in self.base_cols:
            out[c] = out.groupby(self.group_col, sort=False)[c].ffill()
        out[self.base_cols] = out[self.base_cols].fillna(self.medians)

        # Diffs (causal)
        diffs = out.groupby(self.group_col, sort=False)[self.base_cols].diff().fillna(0.0)
        diffs.columns = self.diff_cols

        scaled = pd.concat([out[self.base_cols], diffs], axis=1).astype(np.float32)
        scaled[self.scale_cols] = self.scaler.transform(scaled[self.scale_cols])

        # Write scaled columns and extra features back
        for c in self.base_cols:
            out[c] = scaled[c].astype(np.float32)
        for c in self.diff_cols:
            out[c] = scaled[c].astype(np.float32)
        for c in self.miss_cols:
            out[c] = miss_ind[c].astype(np.float32)

        if self.target_col in out.columns:
            out[self.target_col] = out[self.target_col].fillna(0).astype(np.int32)

        return out

def create_sequences_onset_aware(
    df: pd.DataFrame,
    feature_cols: list,
    n_steps: int = N_STEPS,
    prediction_window: int = PREDICTION_WINDOW,
    target_col: str = TARGET,
    group_col: str = GROUP,
    hour_col: str = HOUR_COL,
):
    """
    Label at time t:
      y_t = 1 if first sepsis onset occurs in next prediction_window hours.
      y_t = 0 otherwise.
    Post-onset timesteps are excluded for septic patients.
    """
    X_list, y_list, groups_list, hours_list = [], [], [], []

    for pid, g in df.groupby(group_col, sort=False):
        g = g.sort_values(hour_col)
        feats = g[feature_cols].to_numpy(dtype=np.float32)
        labels = g[target_col].to_numpy(dtype=np.int32)
        hours = g[hour_col].to_numpy()

        onset_idx_arr = np.where(labels == 1)[0]
        onset_idx = int(onset_idx_arr[0]) if onset_idx_arr.size > 0 else None

        for t in range(len(g)):
            # Exclude post-onset hours from early warning training/evaluation.
            if onset_idx is not None and t >= onset_idx:
                continue

            start = max(0, t - n_steps + 1)
            window = feats[start : t + 1]
            if window.shape[0] < n_steps:
                pad = np.repeat(window[:1], n_steps - window.shape[0], axis=0)
                window = np.vstack([pad, window])

            if onset_idx is None:
                y = 0
            else:
                delta = onset_idx - t
                y = 1 if 1 <= delta <= prediction_window else 0

            X_list.append(window)
            y_list.append(y)
            groups_list.append(pid)
            hours_list.append(hours[t])

    X = np.asarray(X_list, dtype=np.float32)
    y = np.asarray(y_list, dtype=np.int32)
    groups = np.asarray(groups_list)
    hours = np.asarray(hours_list)

    pos = int((y == 1).sum())
    neg = int((y == 0).sum())
    ratio = pos / max(1, len(y))
    print(f"Sequences: {len(y):,} | Pos: {pos:,} | Neg: {neg:,} | Pos ratio: {ratio:.4f}")

    return X, y, groups, hours

def make_dataset(X, y=None, batch_size=BATCH_SIZE, shuffle=False):
    if y is None:
        ds = tf.data.Dataset.from_tensor_slices(X)
    else:
        ds = tf.data.Dataset.from_tensor_slices((X, y))
    if shuffle:
        ds = ds.shuffle(buffer_size=min(SHUFFLE_BUFFER, len(X)), seed=RANDOM_STATE, reshuffle_each_iteration=True)
    return ds.batch(batch_size, drop_remainder=False).prefetch(1)


def build_gru_model(input_shape):
    inp = Input(shape=input_shape)
    x = GRU(64, return_sequences=True, dropout=0.10)(inp)
    x = GRU(32, return_sequences=False, dropout=0.10)(x)
    x = Dense(32, activation="relu")(x)
    x = Dropout(0.20)(x)
    out = Dense(1, activation="sigmoid", dtype="float32")(x)

    model = Model(inp, out)
    model.compile(
        optimizer=Adam(learning_rate=LEARNING_RATE),
        loss="binary_crossentropy",
        metrics=[
            tf.keras.metrics.AUC(curve="ROC", name="roc_auc"),
            tf.keras.metrics.AUC(curve="PR", name="pr_auc"),
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="recall"),
        ],
    )
    return model

def tune_threshold(y_true, probs):
    best_th = 0.5
    best_f1 = -1.0
    for th in np.linspace(0.05, 0.95, 91):
        pred = (probs >= th).astype(np.int32)
        f1 = f1_score(y_true, pred, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_th = float(th)
    return best_th, best_f1


def summarize_metrics(y_true, probs, threshold):
    pred = (probs >= threshold).astype(np.int32)
    metrics = {
        "Accuracy": accuracy_score(y_true, pred),
        "Precision": precision_score(y_true, pred, zero_division=0),
        "Recall": recall_score(y_true, pred, zero_division=0),
        "F1": f1_score(y_true, pred, zero_division=0),
        "AUROC": roc_auc_score(y_true, probs) if len(np.unique(y_true)) > 1 else np.nan,
        "AUPRC": average_precision_score(y_true, probs) if len(np.unique(y_true)) > 1 else np.nan,
        "PredictedPositiveRate": float(pred.mean()),
        "ProbMean": float(probs.mean()),
        "ProbStd": float(probs.std()),
    }
    return metrics

def simulate_patient_live(
    patient_id,
    raw_df,
    preprocessor: CausalPreprocessor,
    model,
    feature_cols,
    n_steps=N_STEPS,
    prediction_window=PREDICTION_WINDOW,
    threshold=0.5,
    group_col=GROUP,
    hour_col=HOUR_COL,
    target_col=TARGET,
):
    patient_raw = raw_df[raw_df[group_col] == patient_id].copy()
    if patient_raw.empty:
        print(f"Patient {patient_id} not found.")
        return pd.DataFrame()

    patient_raw = patient_raw.sort_values(hour_col).reset_index(drop=True)
    patient_proc = preprocessor.transform(patient_raw.copy())

    feats = patient_proc[feature_cols].to_numpy(dtype=np.float32)
    true_labels = patient_raw[target_col].fillna(0).astype(np.int32).to_numpy()
    hours = patient_raw[hour_col].to_numpy()

    onset_idx_arr = np.where(true_labels == 1)[0]
    onset_idx = int(onset_idx_arr[0]) if onset_idx_arr.size > 0 else None
    onset_hour = hours[onset_idx] if onset_idx is not None else None

    risks = []
    future_window_labels = []

    print(f"\n--- Live Simulation: Patient {patient_id} ---")
    for t in range(len(feats)):
        start = max(0, t - n_steps + 1)
        seq = feats[start : t + 1]
        if seq.shape[0] < n_steps:
            pad = np.repeat(seq[:1], n_steps - seq.shape[0], axis=0)
            seq = np.vstack([pad, seq])

        risk = float(model.predict(seq[None, :, :], verbose=0)[0, 0])
        risks.append(risk)

        if onset_idx is None:
            y_future = 0
        else:
            delta = onset_idx - t
            y_future = 1 if 1 <= delta <= prediction_window else 0
        future_window_labels.append(y_future)

        status = "ALERT" if risk >= threshold else "OK"
        print(
            f"Hour {int(hours[t]):3d} | Risk {risk:.3f} | "
            f"Future{prediction_window}hLabel {y_future} | CurrentLabel {int(true_labels[t])} | {status}"
        )

    sim_df = pd.DataFrame(
        {
            "Hour": hours,
            "RiskScore": risks,
            "FutureWindowLabel": future_window_labels,
            "CurrentSepsisLabel": true_labels,
        }
    )

    plt.figure(figsize=(13, 6))
    plt.plot(sim_df["Hour"], sim_df["RiskScore"], label="Predicted Risk", linewidth=2)
    plt.axhline(threshold, linestyle="--", label=f"Threshold ({threshold:.2f})")

    pos_hours = sim_df.loc[sim_df["FutureWindowLabel"] == 1, "Hour"]
    pos_risk = sim_df.loc[sim_df["FutureWindowLabel"] == 1, "RiskScore"]
    if len(pos_hours) > 0:
        plt.scatter(pos_hours, pos_risk, marker="x", s=70, label=f"True +ve (next {prediction_window}h)")

    if onset_hour is not None:
        plt.axvline(onset_hour, color="red", linestyle=":", label=f"Sepsis Onset (hour {int(onset_hour)})")

    plt.title(f"Live Sepsis Risk Simulation (Patient {patient_id})")
    plt.xlabel("Hour")
    plt.ylabel("Risk Probability")
    plt.legend()
    plt.tight_layout()
    plt.show()

    print(
        f"\nSimulation summary | mean={sim_df['RiskScore'].mean():.4f}, "
        f"std={sim_df['RiskScore'].std():.4f}, min={sim_df['RiskScore'].min():.4f}, "
        f"max={sim_df['RiskScore'].max():.4f}"
    )
    return sim_df

def sequence_generator(
    df,
    feature_cols,
    n_steps=N_STEPS,
    prediction_window=PREDICTION_WINDOW,
    target_col=TARGET,
    group_col=GROUP,
    hour_col=HOUR_COL,
    max_samples=None,
):
    """
    Streaming generator yielding (window, label) pairs (window shape: (n_steps, n_features)).
    If max_samples is not None, yields at most max_samples items then returns.
    """
    yielded = 0
    for pid, g in df.groupby(group_col, sort=False):
        g = g.sort_values(hour_col)
        feats = g[feature_cols].to_numpy(dtype=np.float32)  # small copy per patient only
        labels = g[target_col].to_numpy(dtype=np.int32)

        onset_idx_arr = np.where(labels == 1)[0]
        onset_idx = int(onset_idx_arr[0]) if onset_idx_arr.size > 0 else None

        for t in range(len(g)):
            # Exclude post-onset hours
            if onset_idx is not None and t >= onset_idx:
                continue

            start = max(0, t - n_steps + 1)
            window = feats[start : t + 1]
            if window.shape[0] < n_steps:
                pad = np.repeat(window[:1], n_steps - window.shape[0], axis=0)
                window = np.vstack([pad, window])

            if onset_idx is None:
                y = 0
            else:
                delta = onset_idx - t
                y = 1 if 1 <= delta <= prediction_window else 0

            yield window, np.int32(y)

            yielded += 1
            if max_samples is not None and yielded >= max_samples:
                return


def make_tf_dataset_from_df(
    df,
    feature_cols,
    n_steps=N_STEPS,
    prediction_window=PREDICTION_WINDOW,
    batch_size=BATCH_SIZE,
    shuffle=False,
    max_samples=None,
):
    """
    Create a tf.data.Dataset from the streaming generator.
    - For training: set shuffle=True and max_samples=MAX_TRAIN_SAMPLES to cap training set.
    - For val/test: set shuffle=False and max_samples=None.
    """
    out_signature = (
        tf.TensorSpec(shape=(n_steps, len(feature_cols)), dtype=tf.float32),
        tf.TensorSpec(shape=(), dtype=tf.int32),
    )

    ds = tf.data.Dataset.from_generator(
        lambda: sequence_generator(
            df,
            feature_cols,
            n_steps=n_steps,
            prediction_window=prediction_window,
            max_samples=max_samples,
        ),
        output_signature=out_signature,
    )
    if shuffle:
        # smaller buffer uses less RAM — set to a fraction of MAX_TRAIN_SAMPLES if capped
        buf = min(int(SHUFFLE_BUFFER), max_samples if max_samples is not None else int(SHUFFLE_BUFFER))
        ds = ds.shuffle(buffer_size=max(1024, buf), seed=RANDOM_STATE, reshuffle_each_iteration=True)
    ds = ds.batch(batch_size, drop_remainder=False).prefetch(tf.data.AUTOTUNE)
    return ds

def main():
    set_seed(RANDOM_STATE)

    tf.keras.backend.clear_session()
    gc.collect()
    mixed_precision.set_global_policy("mixed_float16")


    gpus = tf.config.experimental.list_physical_devices("GPU")
    if gpus:
        try:
            for gpu in gpus:
                tf.config.experimental.set_memory_growth(gpu, True)
            print(f"GPU(s) detected: {len(gpus)}")
        except Exception as e:
            print("GPU config error:", e)
    else:
        print("No GPU found. Running on CPU.")

    print("Loading data...")
    df = pd.read_csv(CSV_PATH)
    if "Unnamed: 0" in df.columns:
        df = df.drop(columns=["Unnamed: 0"])

    # ----------------------------
    # Reduce number of non-septic patients
    # ----------------------------

    print("\nReducing number of non-septic patients...")

    # Determine which patients develop sepsis
    patient_sepsis = df.groupby(GROUP)[TARGET].max()

    # Patient IDs
    septic_ids = patient_sepsis[patient_sepsis == 1].index.values
    non_septic_ids = patient_sepsis[patient_sepsis == 0].index.values

    print(f"Septic patients: {len(septic_ids)}")
    print(f"Non-septic patients: {len(non_septic_ids)}")

    # Keep ALL septic patients
    # Keep only N times more non-septic patients
    NEGATIVE_MULTIPLIER = 4

    np.random.seed(RANDOM_STATE)

    num_non_septic_to_keep = min(
        len(non_septic_ids),
        NEGATIVE_MULTIPLIER * len(septic_ids)
    )

    keep_non_septic = np.random.choice(
        non_septic_ids,
        size=num_non_septic_to_keep,
        replace=False
    )

    # Combine patient IDs to keep
    patients_to_keep = np.concatenate([septic_ids, keep_non_septic])

    # Filter dataset
    df = df[df[GROUP].isin(patients_to_keep)].reset_index(drop=True)

    print(f"Patients after reduction: {len(patients_to_keep)}")
    print(f"Rows after reduction: {len(df):,}")
    print(f"New positive rate: {df[TARGET].mean():.4f}")


    # Basic cleanup
    df = df.dropna(subset=[GROUP, TARGET, HOUR_COL]).copy()
    df[GROUP] = df[GROUP].astype(np.int64)
    df[TARGET] = df[TARGET].astype(np.int32)
    df = df.sort_values([GROUP, HOUR_COL]).reset_index(drop=True)

    print(f"Rows: {len(df):,}, Columns: {df.shape[1]}, Patients: {df[GROUP].nunique():,}")
    print(f"Overall raw positive rate ({TARGET}=1): {df[TARGET].mean():.4f}")

    # ----------------------------
    # Group split (patient-level)
    # ----------------------------
    gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
    train_idx, test_idx = next(gss.split(df, groups=df[GROUP].values))
    df_train_raw = df.iloc[train_idx].reset_index(drop=True)
    df_test_raw = df.iloc[test_idx].reset_index(drop=True)

    gss2 = GroupShuffleSplit(
        n_splits=1,
        test_size=VAL_SIZE / (1 - TEST_SIZE),
        random_state=RANDOM_STATE,
    )
    tr_idx, val_idx = next(gss2.split(df_train_raw, groups=df_train_raw[GROUP].values))
    df_tr_raw = df_train_raw.iloc[tr_idx].reset_index(drop=True)
    df_val_raw = df_train_raw.iloc[val_idx].reset_index(drop=True)

    print(
        f"Split patients | train={df_tr_raw[GROUP].nunique()}, "
        f"val={df_val_raw[GROUP].nunique()}, test={df_test_raw[GROUP].nunique()}"
    )

    # ----------------------------
    # Fit causal preprocessing on train only
    # ----------------------------
    preprocessor = CausalPreprocessor(
        target_col=TARGET,
        group_col=GROUP,
        hour_col=HOUR_COL,
        max_col_missing=MAX_COL_MISSING,
    )
    preprocessor.fit(df_tr_raw)
    feature_cols = preprocessor.feature_cols
    print(f"Feature count after engineering: {len(feature_cols)}")

    df_tr = preprocessor.transform(df_tr_raw)
    df_val = preprocessor.transform(df_val_raw)
    df_test = preprocessor.transform(df_test_raw)

    # ----------------------------
    # Build onset-aware sequences
    # ----------------------------
    print("\nCreating train sequences...")
    X_tr, y_tr, groups_tr, _ = create_sequences_onset_aware(
        df_tr,
        feature_cols,
        n_steps=N_STEPS,
        prediction_window=PREDICTION_WINDOW,
    )

    if len(y_tr) > MAX_TRAIN_SAMPLES:
      rs = np.random.RandomState(RANDOM_STATE)
      keep = rs.choice(len(y_tr), size=MAX_TRAIN_SAMPLES, replace=False)
      X_tr = X_tr[keep]
      y_tr = y_tr[keep]
      groups_tr = groups_tr[keep]
      print(f"Capped train samples to {len(y_tr):,}")


    print("Creating val sequences...")
    X_val, y_val, groups_val, _ = create_sequences_onset_aware(
        df_val,
        feature_cols,
        n_steps=N_STEPS,
        prediction_window=PREDICTION_WINDOW,
    )

    print("Creating test sequences...")
    X_test, y_test, groups_test, _ = create_sequences_onset_aware(
        df_test,
        feature_cols,
        n_steps=N_STEPS,
        prediction_window=PREDICTION_WINDOW,
    )

    if (y_tr == 1).sum() == 0:
        raise RuntimeError("No positive samples in training sequences. Check label generation.")

    # ----------------------------
    # Class weighting (no negative random dropping)
    # ----------------------------
    classes = np.unique(y_tr)
    cw = compute_class_weight(class_weight="balanced", classes=classes, y=y_tr)
    class_weight = {int(c): float(w) for c, w in zip(classes, cw)}
    if 0 not in class_weight:
        class_weight[0] = 1.0
    if 1 not in class_weight:
        class_weight[1] = 1.0
    print("Class weight:", class_weight)

    # ----------------------------
    # Train GRU
    # ----------------------------
    model = build_gru_model(input_shape=(N_STEPS, len(feature_cols)))
    train_ds = make_dataset(X_tr, y_tr, batch_size=BATCH_SIZE, shuffle=True)
    val_ds = make_dataset(X_val, y_val, batch_size=BATCH_SIZE, shuffle=False)
    test_ds = make_dataset(X_test, y_test, batch_size=BATCH_SIZE, shuffle=False)

    callbacks = [
        EarlyStopping(monitor="val_pr_auc", mode="max", patience=PATIENCE, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor="val_pr_auc", mode="max", factor=0.5, patience=2, min_lr=1e-5, verbose=1),
    ]

    del df_tr_raw, df_val_raw
    gc.collect()

    print("\nTraining GRU model...")
    _ = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=callbacks,
        class_weight=class_weight,
        verbose=1,
    )

    # ----------------------------
    # Validation threshold tuning
    # ----------------------------
    val_probs = model.predict(val_ds, verbose=0).ravel()
    best_thresh, best_f1 = tune_threshold(y_val, val_probs)
    print(f"\nBest validation threshold: {best_thresh:.3f} (F1={best_f1:.4f})")
    print(f"Validation probability distribution: mean={val_probs.mean():.4f}, std={val_probs.std():.4f}")

    # ----------------------------
    # Test metrics
    # ----------------------------
    test_probs = model.predict(test_ds, verbose=0).ravel()
    test_metrics = summarize_metrics(y_test, test_probs, threshold=best_thresh)
    print("\nTest Metrics")
    for k, v in test_metrics.items():
        print(f"{k:>22}: {v:.4f}" if isinstance(v, float) else f"{k:>22}: {v}")

    # ----------------------------
    # Live simulation for a septic patient
    # ----------------------------
    septic_patient_ids = (
        df_test_raw.groupby(GROUP, sort=False)[TARGET].max().loc[lambda s: s == 1].index.tolist()
    )
    if not septic_patient_ids:
        print("No septic patient found in test split for live simulation.")
        return

    selected_patient = septic_patient_ids[0]
    print(f"\nSelected septic patient for simulation: {selected_patient}")

    sim_df = simulate_patient_live(
        patient_id=selected_patient,
        raw_df=df_test_raw,  # raw split; preprocessing applied inside function exactly once
        preprocessor=preprocessor,
        model=model,
        feature_cols=feature_cols,
        n_steps=N_STEPS,
        prediction_window=PREDICTION_WINDOW,
        threshold=best_thresh,
    )

    # Optional: save simulation output
    sim_df.to_csv("simulation_results_updated.csv", index=False)
    print("\nSaved simulation output to simulation_results_updated.csv")

if __name__ == "__main__":
    main()